# 06 — Composition-group bootstrap confidence intervals
**Inputs:** matching final holdouts and split-table composition groups. **Outputs:** metric summaries and resumable checkpoints under `results/paper/`, with the combined `bootstrap_summary.csv` under COD_CANON_V2.

The PAPER_V2 selection covers RF and LightGBM for COD-I/O/M and MP-I: 24 metric jobs. MP-O/M have point estimates but no bootstrap intervals from this protocol.

The bootstrap unit is the composition group (`group_id`), not the individual CIF record.

Folds are split on `group_id`. A group is one composition and can contain several structures; the centrosymmetry label does not vary inside a group. Each replicate draws groups with replacement and keeps every record that belongs to a drawn group.

`test_value` is the metric on the full test set. `ci_low` and `ci_high` are the 2.5 and 97.5 percentiles of 2000 replicates (seed 42) and are stored separately from that point estimate. MCC and BACC use threshold 0.5. A finished job is skipped unless `FORCE` is true. An interrupted job resumes from the last saved replicate. `resume=False` refuses to ignore an existing checkpoint.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve imports relative to the source directory.
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "project_config.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


from project_config import make_cfg
from private_paths import MP_DIR, COD_DIR
from core.notebook_logger import NotebookLogger
from reports.bootstrap_auc import bootstrap_metric_from_holdouts

cfg_mp = make_cfg(project_name="MP_ALL", root_cif=MP_DIR)
cfg_cod = make_cfg(project_name="COD_CANON_V2", root_cif=COD_DIR)

PAPER_RUN = "PAPER_V2"
RUN_OVERRIDES = {}
runs = {model: f"{PAPER_RUN}__final__{model}__magpie" for model in ("rf", "lgbm")}
from reports.paper_inputs import prepare_report_inputs
prepare_report_inputs([cfg_cod, cfg_mp], runs, kind="holdout",
                      paper_run=PAPER_RUN, overrides=RUN_OVERRIDES)

N_BOOT = 2000
SEED = 42
THRESHOLD = 0.5
FORCE = False

metrics_to_eval = [
    ("auc", {}),
    ("mcc", {"threshold": THRESHOLD}),
    ("bacc", {"threshold": THRESHOLD}),
]

logger = NotebookLogger(project_dir=cfg_cod.PROJECT_DIR, prefix="bootstrap_PAPER_V2")
logger.start(header_lines=[
    "Project COD    : COD_CANON_V2",
    "Project MP     : MP_ALL",
    "Runs           : PAPER_V2 final rf/lgbm magpie",
    "Bootstrap unit : group_id (composition groups)",
    f"Replicates     : {N_BOOT}",
    f"Seed           : {SEED}",
    f"MCC/BACC thr   : {THRESHOLD}",
    f"Force          : {FORCE}",
])

summaries = []
try:
    for metric, extra in metrics_to_eval:
        summ_cod = bootstrap_metric_from_holdouts(
            cfg_cod,
            metric=metric,
            runs=runs,
            models=["rf", "lgbm"],
            domains=["I", "O", "M"],
            n_boot=N_BOOT,
            seed=SEED,
            save=True,
            project_dir=cfg_cod.PROJECT_DIR,
            save_run_id="paper",
            force=FORCE,
            resume=True,
            **extra,
        )

        summ_mp = bootstrap_metric_from_holdouts(
            cfg_mp,
            metric=metric,
            runs=runs,
            models=["rf", "lgbm"],
            domains=["I"],
            n_boot=N_BOOT,
            seed=SEED,
            save=True,
            project_dir=cfg_mp.PROJECT_DIR,
            save_run_id="paper",
            force=FORCE,
            resume=True,
            **extra,
        )

        summaries.append(pd.concat([summ_cod, summ_mp], ignore_index=True))

    results_table = pd.concat(summaries, ignore_index=True)
    print("\nSummary (all projects/models/domains/metrics):")
    display(results_table)

    paper_cols = [
        "project", "model", "run_id", "domain", "metric", "threshold",
        "test_value", "ci_low", "ci_high", "n_records", "n_groups", "unit",
    ]
    cols = [c for c in paper_cols if c in results_table.columns]
    paper_summary = results_table[cols].copy()
    print("\nSummary (full test and CI bounds):")
    display(paper_summary)
finally:
    logger.stop()


## Combined table
Store full-test estimates, percentile endpoints and bootstrap diagnostics for subsequent publication export.


In [ ]:

# Combined table: full-test metric and CI bounds for COD_CANON_V2 and MP_ALL.
cod_results_dir = Path(cfg_cod.RESULTS_DIR) / "paper" / "tables"
cod_results_dir.mkdir(parents=True, exist_ok=True)

output_path = cod_results_dir / "bootstrap_summary.csv"
results_table.to_csv(output_path, index=False)
print(f"Saved bootstrap summary to: {output_path}")
